In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By

import time
import csv

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
driver_path = "C:/studies/uni/semester 4/Machine Learning/lab works/lb 1/chromedriver-win64/chromedriver.exe"
service = webdriver.ChromeService(executable_path=driver_path)
driver = webdriver.Chrome(service=service)

## Scrapping `https://www.kaggle.com/datasets?tags=12107-Computer+Science`

In [ ]:
KAGGLE_URL = "https://www.kaggle.com/datasets?tags=12107-Computer+Science"

In [ ]:
def parse_dataset(dataset):
    name = dataset.find_element(By.XPATH, './/div[@class="sc-kCuUfV sc-bjxVRI bXQAUF dEeLno"]')
    author = dataset.find_element(By.XPATH, './/a[@class="sc-kjwdDK fDGKWq"]')
    upvotes = dataset.find_element(By.XPATH, './/span[@class="sc-fVHBlr sc-kJNXNN gHmzEf rvJZo"]')
    try:
        usability = dataset.find_element(By.XPATH, './/span[@class="sc-kMemMU SkXLN"]')
    except:
        usability = dataset.find_element(By.XPATH, './/span[@class="sc-kMemMU duSZa-d"]')

    return [name.text, author.text, usability.text, upvotes.text]

def scrape_datasets():
    result = [
        ["Dataset name", "author", "usabilities", "upvotes"],
    ]

    for i in range(1, 26):
        driver.get(KAGGLE_URL + f"&page={i}")
        time.sleep(3)
        datasets = driver.find_elements(By.XPATH, '//div[@class="sc-eXVaYZ idRORJ km-listitem--large"]')
        for dataset in datasets:
            result.append(parse_dataset(dataset))

    driver.quit()
    return result

In [ ]:
path = 'datasets.csv'
data = scrape_datasets()
with open(path, 'w', newline='', encoding="utf-8") as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerows(data)

### Processing datasets data

In [ ]:
datasets_df = pd.read_csv("datasets.csv")
datasets_df.head()

In [ ]:
datasets_df["name_len"] = datasets_df['Dataset name'].astype(str).str.len()
datasets_df[["Dataset name","author","usabilities", "name_len"]].head()

In [ ]:
datasets_clean = datasets_df.dropna(subset=["usabilities"]).copy()
datasets_clean.isna().sum()
datasets_clean[["usabilities", "name_len"]].describe()

In [ ]:
stats_datasets = {
    "usabilities_mean": datasets_clean["usabilities"].mean(),
    "usabilities_median": datasets_clean["usabilities"].median(),
    "usabilities_mode": datasets_clean["usabilities"].mode().iloc[0],
    "usabilities_std": datasets_clean["usabilities"].std(),

    "name_len_mean": datasets_clean["name_len"].mean(),
    "name_len_median": datasets_clean["name_len"].median(),
    "name_len_mode": datasets_clean["name_len"].mode().iloc[0],
    "name_len_std": datasets_clean["name_len"].std(),
}
stats_datasets

In [ ]:
plt.figure()
plt.boxplot(datasets_df["usabilities"].dropna())
plt.title("Datasets: Usability (Box Plot)")
plt.ylabel("Usability")
plt.show()

In [ ]:
author_counts = datasets_df["author"].value_counts().head(10)
plt.figure()
plt.bar(author_counts.index, author_counts.values)
plt.title("Top 10 Authors by Number of Datasets")
plt.xlabel("Author")
plt.ylabel("Number of Datasets")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure()
plt.scatter(datasets_df["usabilities"], datasets_df["upvotes"])
plt.title("Usability vs Upvotes")
plt.xlabel("Usability")
plt.ylabel("Upvotes")
plt.show()

## Scrapping `https://books.toscrape.com/`

In [ ]:
URL = "https://books.toscrape.com/"
CATALOGUE_URL = URL + "catalogue/"

In [ ]:
def parse_book(book):
    title = book.find_element(By.XPATH, './/h3')
    rating = book.find_element(By.XPATH, './/p[contains(@class,"star-rating")]').get_attribute("class").split()[1]
    price = book.find_element(By.XPATH, './/p[@class="price_color"]')

    return [title.text, price.text, rating]

def scrape_books():
    driver.get(URL)
    results = [
        ["title", "price", "rating"],
    ]
    while len(results) <= 501:
        books = driver.find_elements(By.XPATH, '//article[@class="product_pod"]')
        for book in books:
            results.append(parse_book(book))
            if len(results) >= 501:
                break
        driver.find_element(By.XPATH, '//a[text()="next"]').click()

    driver.quit()
    return results

In [ ]:
data = scrape_books()
path = 'books.csv'
with open(path, 'w', newline='', encoding="utf-8") as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerows(data)

### Processing books data

In [ ]:
books_df = pd.read_csv("books.csv")
books_df.head()

In [ ]:
books_df["price_num"] = (books_df["price"].astype(str).str.replace(r"[^\d.]", "", regex=True))
books_df["price_num"] = pd.to_numeric(books_df["price_num"], errors="coerce")

In [ ]:
rating_map = {"One":1, "Two":2, "Three":3, "Four":4, "Five":5}
books_df["rating_num"] = books_df["rating"].map(rating_map)

In [ ]:
books_clean = books_df.dropna(subset=["price_num", "rating_num"]).copy()
books_clean.isna().sum()

In [ ]:
books_clean[["price_num", "rating_num"]].describe()

In [ ]:

stats_datasets = {
    "price_mean": books_clean["price_num"].mean(),
    "price_median": books_clean["price_num"].median(),
    "price_mode": books_clean["price_num"].mode().iloc[0] if not books_clean["price_num"].mode().empty else np.nan,
    "price_std": books_clean["price_num"].std(),

    "rating_mean": books_clean["rating_num"].mean(),
    "rating_median": books_clean["rating_num"].median(),
    "rating_mode": books_clean["rating_num"].mode().iloc[0] if not books_clean["rating_num"].mode().empty else np.nan,
    "rating_std": books_clean["rating_num"].std(),
}
stats_datasets

In [ ]:
plt.figure()
plt.boxplot(books_df["price_num"].dropna())
plt.title("Books: Price discribution (Box Plot)")
plt.ylabel("Price")
plt.show()


In [ ]:
rating_counts = books_df["rating_num"].value_counts()
plt.figure()
plt.bar(rating_counts.index, rating_counts.values)
plt.title("Books: Rating distribution")
plt.xlabel("Rating")
plt.ylabel("Number")
plt.show()


In [ ]:
plt.figure()
plt.scatter(books_df["price_num"], books_df["rating_num"])
plt.title("Books: Prices vs Ratings")
plt.xlabel("Price")
plt.ylabel("Rating")
plt.show()